# `dim_indicator`: подробный EDA и очистка данных

Этот notebook посвящён таблице `dim_indicator`.

В ней есть индикаторы **из двух источников**:

- `un`
- `worldbank`

Это важно, потому что теперь `dim_indicator` — не просто маленький справочник кодов, а уже **единый словарь метрик проекта**, который связывает:

- демографические и репродуктивные UN indicators,
- socioeconomic и health indicators World Bank.

## Зачем нужна таблица `dim_indicator`

Она нужна для того, чтобы:

1. расшифровывать технические коды индикаторов;
2. понимать, из какого источника пришёл показатель;
3. строить понятные подписи для графиков и фронтенда;
4. проверять корректность `indicator_code` в `fact_indicator_value`;
5. формировать semantic layer для всего проекта.

## Что мы будем делать

1. Загрузим таблицу  
2. Разберём смысл колонок  
3. Подробно расшифруем все индикаторы  
4. Проверим пропуски  
5. Проверим дубликаты  
6. Проверим распределение по источникам  
7. Построим `dim_indicator_clean`  
8. Построим `dim_indicator_enriched`  
9. Сохраним результаты

In [1]:
from pathlib import Path
import pandas as pd
import plotly.express as px

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

DATA_DIR = Path("./data_exports")
OUTPUT_DIR = Path("./notebooks/eda_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Загрузка данных

Таблица читается из:

- `./data_exports/dim_indicator.csv`

In [2]:
dim_indicator = pd.read_csv(DATA_DIR / "dim_indicator.csv")
print("dim_indicator:", dim_indicator.shape)
dim_indicator.head(20)

dim_indicator: (17, 4)


,source,code,name,unit
0,worldbank,SP.DYN.TFRT.IN,"Fertility rate, total (births per woman)",NaN
1,worldbank,SP.ADO.TFRT,"Adolescent fertility rate (births per 1,000 wo...",NaN
2,worldbank,SH.STA.MMRT.NE,"Maternal mortality ratio (per 100,000 live bir...",NaN
3,worldbank,SL.TLF.CACT.FE.ZS,"Labor force participation rate, female (%)",NaN
4,worldbank,NY.GDP.PCAP.CD,GDP per capita (current US$),NaN
5,worldbank,SP.URB.TOTL.IN.ZS,Urban population (% of total),NaN
6,worldbank,SE.SEC.ENRR.FE,"School enrollment, secondary, female (%)",NaN
7,worldbank,SH.XPD.CHEX.GD.ZS,Current health expenditure (% of GDP),NaN
8,un,UN_18,Mean age of childbearing (5-year),NaN
9,un,UN_67,Median age of population,NaN


## 2. Что означают колонки таблицы

### `source`
Источник данных.

Это особенно важно, потому что в таблице смешаны:
- `un`
- `worldbank`

Значит, один только `code` недостаточно считать уникальным.  
Естественный ключ таблицы:

**(`source`, `code`)**

### `code`
Технический код индикатора.

Примеры:
- `UN_18`
- `UN_86`
- `SP.DYN.TFRT.IN`
- `NY.GDP.PCAP.CD`

### `name`
Полное название индикатора.  
Именно это поле будет полезно для графиков, подписей, tooltip и интерпретации.

### `unit`
Единица измерения.  
Если в clean-версии проекта она не используется, её можно удалить.

## 3. Подробная расшифровка индикаторов

Ниже разберём индикаторы по источникам.

# 3.1 Индикаторы `un`

### `UN_18`
**Mean age of childbearing (5-year)**

Что показывает:
- средний возраст деторождения.

Почему важен:
- помогает понять, в каком возрасте в среднем происходят рождения;
- полезен для анализа демографического перехода и изменения reproductive patterns.

### `UN_2`
**Contraceptive prevalence: Any modern method (Percent)**

Что показывает:
- распространённость современных методов контрацепции.

Почему важен:
- один из самых сильных indicators для связи policy, family planning и fertility outcomes;
- тематически очень хорошо ложится на ваш проект.

### `UN_4`
**Unmet need for family planning: Any method (Percent)**

Что показывает:
- долю женщин, у которых есть неудовлетворённая потребность в семейном планировании.

Почему важен:
- это очень сильный reproductive health / access indicator;
- помогает объяснять различия между policy environment и реальным доступом к услугам.

### `UN_41`
**Female population of reproductive age (15-49 years)**

Что показывает:
- численность женского населения репродуктивного возраста.

Почему важен:
- useful demographic context indicator;
- может быть полезен как базовая структура населения, но не как headline metric.

### `UN_50`
**Population Change**

Что показывает:
- изменение численности населения.

Почему важен:
- помогает описывать общий демографический контекст;
- полезен как макро-level background indicator.

### `UN_66`
**Crude rate of net migration**

Что показывает:
- грубый коэффициент чистой миграции.

Почему важен:
- помогает учитывать вклад миграции в демографическую динамику;
- полезен как дополнительная объясняющая переменная.

### `UN_67`
**Median age of population**

Что показывает:
- медианный возраст населения.

Почему важен:
- очень хороший indicator стадии демографического перехода;
- помогает показывать ageing / population structure context.

### `UN_83`
**Child dependency ratio**

Что показывает:
- коэффициент детской демографической нагрузки.

Почему важен:
- связан со структурой населения;
- помогает анализировать возрастную композицию общества.

### `UN_86`
**Total dependency ratio**

Что показывает:
- общий коэффициент демографической нагрузки.

Почему важен:
- useful macro-demographic structure indicator;
- хорош для общего контекста population structure.

# 3.2 Индикаторы `worldbank`

### `NY.GDP.PCAP.CD`
**GDP per capita (current US$)**  
Экономический контекст страны.

### `SE.SEC.ENRR.FE`
**School enrollment, secondary, female (%)**  
Индикатор женского образования.

### `SH.STA.MMRT.NE`
**Maternal mortality ratio (per 100,000 live births)**  
Один из главных health outcome indicators проекта. Коэффициент материнской смертности.

### `SH.XPD.CHEX.GD.ZS`
**Current health expenditure (% of GDP)**  
Контекстный health system indicator. Текущие расходы на здравоохранение

### `SL.TLF.CACT.FE.ZS`
**Labor force participation rate, female (%)**  
Индикатор экономической активности женщин.

### `SP.ADO.TFRT`
**Adolescent fertility rate (births per 1,000 women ages 15-19)**  
Один из ключевых outcome indicators проекта. Коэффициент подростковой фертильности

### `SP.DYN.TFRT.IN`
**Fertility rate, total (births per woman)**  
Главный демографический индикатор проекта.

### `SP.URB.TOTL.IN.ZS`
**Urban population (% of total)**  
Контекстный индикатор урбанизации.

## 4. Первичный обзор таблицы

In [3]:
dim_indicator.head(20)

,source,code,name,unit
0,worldbank,SP.DYN.TFRT.IN,"Fertility rate, total (births per woman)",NaN
1,worldbank,SP.ADO.TFRT,"Adolescent fertility rate (births per 1,000 wo...",NaN
2,worldbank,SH.STA.MMRT.NE,"Maternal mortality ratio (per 100,000 live bir...",NaN
3,worldbank,SL.TLF.CACT.FE.ZS,"Labor force participation rate, female (%)",NaN
4,worldbank,NY.GDP.PCAP.CD,GDP per capita (current US$),NaN
5,worldbank,SP.URB.TOTL.IN.ZS,Urban population (% of total),NaN
6,worldbank,SE.SEC.ENRR.FE,"School enrollment, secondary, female (%)",NaN
7,worldbank,SH.XPD.CHEX.GD.ZS,Current health expenditure (% of GDP),NaN
8,un,UN_18,Mean age of childbearing (5-year),NaN
9,un,UN_67,Median age of population,NaN


In [4]:
dim_indicator.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17 entries, 0 to 16
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   source  17 non-null     object 
 1   code    17 non-null     object 
 2   name    17 non-null     object 
 3   unit    0 non-null      float64
dtypes: float64(1), object(3)
memory usage: 676.0+ bytes


## 5. Первичная нормализация строковых полей

Приводим строки к аккуратному виду:
- убираем пробелы по краям;
- схлопываем повторяющиеся пробелы;
- `source` переводим в lower case;
- `code` переводим в upper case.

In [5]:
dim_indicator_work = dim_indicator.copy()

def clean_string_series(s: pd.Series) -> pd.Series:
    return (
        s.astype("string")
         .str.strip()
         .str.replace(r"\s+", " ", regex=True)
    )

for col in dim_indicator_work.columns:
    dim_indicator_work[col] = clean_string_series(dim_indicator_work[col])

if "source" in dim_indicator_work.columns:
    dim_indicator_work["source"] = dim_indicator_work["source"].str.lower()

if "code" in dim_indicator_work.columns:
    dim_indicator_work["code"] = dim_indicator_work["code"].str.upper()

dim_indicator_work.head(20)

,source,code,name,unit
0,worldbank,SP.DYN.TFRT.IN,"Fertility rate, total (births per woman)",<NA>
1,worldbank,SP.ADO.TFRT,"Adolescent fertility rate (births per 1,000 wo...",<NA>
2,worldbank,SH.STA.MMRT.NE,"Maternal mortality ratio (per 100,000 live bir...",<NA>
3,worldbank,SL.TLF.CACT.FE.ZS,"Labor force participation rate, female (%)",<NA>
4,worldbank,NY.GDP.PCAP.CD,GDP per capita (current US$),<NA>
5,worldbank,SP.URB.TOTL.IN.ZS,Urban population (% of total),<NA>
6,worldbank,SE.SEC.ENRR.FE,"School enrollment, secondary, female (%)",<NA>
7,worldbank,SH.XPD.CHEX.GD.ZS,Current health expenditure (% of GDP),<NA>
8,un,UN_18,Mean age of childbearing (5-year),<NA>
9,un,UN_67,Median age of population,<NA>


## 6. Анализ пропусков

In [6]:
missing_summary = pd.DataFrame({
    "missing_count": dim_indicator_work.isna().sum(),
    "missing_ratio": dim_indicator_work.isna().mean()
}).sort_values("missing_ratio", ascending=False)

missing_summary

,missing_count,missing_ratio
unit,17,1.0
source,0,0.0
code,0,0.0
name,0,0.0


In [7]:
fig = px.bar(
    missing_summary.reset_index().rename(columns={"index": "column"}),
    x="column",
    y="missing_ratio",
    title="Доля пропусков по колонкам в dim_indicator",
    text="missing_ratio"
)
fig.update_traces(texttemplate="%{text:.2%}", textposition="outside")
fig.update_layout(xaxis_title="Колонка", yaxis_title="Доля пропусков")
fig.show()

## 7. Проверка дубликатов

Для `dim_indicator` естественный ключ:

- `source`
- `code`

In [8]:
full_duplicates = dim_indicator_work[dim_indicator_work.duplicated(keep=False)].copy()
print("Количество строк, участвующих в полных дублях:", len(full_duplicates))
full_duplicates

Количество строк, участвующих в полных дублях: 0


,source,code,name,unit


In [9]:
key_dups = dim_indicator_work[
    dim_indicator_work.duplicated(subset=["source", "code"], keep=False)
].copy()

print("Количество строк, участвующих в дублях по (source, code):", len(key_dups))
key_dups

Количество строк, участвующих в дублях по (source, code): 0


,source,code,name,unit


In [10]:
dup_group_summary = (
    dim_indicator_work.groupby(["source", "code"], dropna=False)
    .agg(
        n_rows=("name", "size"),
        n_unique_names=("name", pd.Series.nunique)
    )
    .reset_index()
    .sort_values(["n_rows", "n_unique_names"], ascending=False)
)

dup_group_summary[dup_group_summary["n_rows"] > 1]

,source,code,n_rows,n_unique_names


## 8. Распределение индикаторов по источникам

Теперь особенно важно посмотреть, сколько индикаторов приходит из каждого источника.

In [11]:
source_counts = (
    dim_indicator_work["source"]
    .value_counts(dropna=False)
    .rename_axis("source")
    .reset_index(name="n_indicators")
)

source_counts

,source,n_indicators
0,un,9
1,worldbank,8


In [12]:
fig = px.bar(
    source_counts,
    x="source",
    y="n_indicators",
    title="Количество индикаторов по источникам",
    text="n_indicators"
)
fig.update_layout(xaxis_title="Источник", yaxis_title="Количество индикаторов")
fig.show()

## 9. Проверка кодов и названий индикаторов

In [13]:
dim_indicator_work[["source", "code", "name"]]

,source,code,name
0,worldbank,SP.DYN.TFRT.IN,"Fertility rate, total (births per woman)"
1,worldbank,SP.ADO.TFRT,"Adolescent fertility rate (births per 1,000 wo..."
2,worldbank,SH.STA.MMRT.NE,"Maternal mortality ratio (per 100,000 live bir..."
3,worldbank,SL.TLF.CACT.FE.ZS,"Labor force participation rate, female (%)"
4,worldbank,NY.GDP.PCAP.CD,GDP per capita (current US$)
5,worldbank,SP.URB.TOTL.IN.ZS,Urban population (% of total)
6,worldbank,SE.SEC.ENRR.FE,"School enrollment, secondary, female (%)"
7,worldbank,SH.XPD.CHEX.GD.ZS,Current health expenditure (% of GDP)
8,un,UN_18,Mean age of childbearing (5-year)
9,un,UN_67,Median age of population


На этом этапе полезно глазами проверить:
- нет ли неожиданных кодов;
- нет ли лишних пробелов;
- выглядят ли названия осмысленно;
- нет ли повторов по смыслу.

## 10. Построение `dim_indicator_clean`

Правила очистки:

1. удаляем строки без `source` и `code`;  
2. удаляем полные дубли;  
3. удаляем дубли по `(source, code)`;  
4. при необходимости удаляем `unit`.

In [14]:
dim_indicator_clean = dim_indicator_work.copy()

dim_indicator_clean = dim_indicator_clean.dropna(subset=["source", "code"]).copy()
dim_indicator_clean = dim_indicator_clean.drop_duplicates().copy()
dim_indicator_clean = dim_indicator_clean.drop_duplicates(subset=["source", "code"], keep="first").copy()

if "unit" in dim_indicator_clean.columns:
    dim_indicator_clean = dim_indicator_clean.drop(columns=["unit"]).copy()

dim_indicator_clean

,source,code,name
0,worldbank,SP.DYN.TFRT.IN,"Fertility rate, total (births per woman)"
1,worldbank,SP.ADO.TFRT,"Adolescent fertility rate (births per 1,000 wo..."
2,worldbank,SH.STA.MMRT.NE,"Maternal mortality ratio (per 100,000 live bir..."
3,worldbank,SL.TLF.CACT.FE.ZS,"Labor force participation rate, female (%)"
4,worldbank,NY.GDP.PCAP.CD,GDP per capita (current US$)
5,worldbank,SP.URB.TOTL.IN.ZS,Urban population (% of total)
6,worldbank,SE.SEC.ENRR.FE,"School enrollment, secondary, female (%)"
7,worldbank,SH.XPD.CHEX.GD.ZS,Current health expenditure (% of GDP)
8,un,UN_18,Mean age of childbearing (5-year)
9,un,UN_67,Median age of population


## 11. Построение `dim_indicator_enriched`

Добавим проектные поля:
- `project_label`
- `theme_group`
- `narrative_role`

In [15]:
indicator_enrichment = {
    "UN_18": {
        "project_label": "Mean age of childbearing",
        "theme_group": "fertility_timing",
        "narrative_role": "context"
    },
    "UN_2": {
        "project_label": "Modern contraceptive prevalence",
        "theme_group": "family_planning",
        "narrative_role": "core_context"
    },
    "UN_4": {
        "project_label": "Unmet need for family planning",
        "theme_group": "family_planning",
        "narrative_role": "core_context"
    },
    "UN_41": {
        "project_label": "Female population 15-49",
        "theme_group": "population_structure",
        "narrative_role": "context"
    },
    "UN_50": {
        "project_label": "Population change",
        "theme_group": "population_dynamics",
        "narrative_role": "context"
    },
    "UN_66": {
        "project_label": "Crude net migration rate",
        "theme_group": "migration",
        "narrative_role": "context"
    },
    "UN_67": {
        "project_label": "Median age",
        "theme_group": "population_structure",
        "narrative_role": "context"
    },
    "UN_83": {
        "project_label": "Child dependency ratio",
        "theme_group": "population_structure",
        "narrative_role": "context"
    },
    "UN_86": {
        "project_label": "Total dependency ratio",
        "theme_group": "population_structure",
        "narrative_role": "context"
    },
    "NY.GDP.PCAP.CD": {
        "project_label": "GDP per capita",
        "theme_group": "economy",
        "narrative_role": "context"
    },
    "SE.SEC.ENRR.FE": {
        "project_label": "Female secondary enrollment",
        "theme_group": "education",
        "narrative_role": "gender_context"
    },
    "SH.STA.MMRT.NE": {
        "project_label": "Maternal mortality",
        "theme_group": "health",
        "narrative_role": "health_outcome"
    },
    "SH.XPD.CHEX.GD.ZS": {
        "project_label": "Health expenditure (% GDP)",
        "theme_group": "health_system",
        "narrative_role": "context"
    },
    "SL.TLF.CACT.FE.ZS": {
        "project_label": "Female labor force participation",
        "theme_group": "labor_gender",
        "narrative_role": "gender_context"
    },
    "SP.ADO.TFRT": {
        "project_label": "Adolescent fertility",
        "theme_group": "fertility",
        "narrative_role": "core_outcome"
    },
    "SP.DYN.TFRT.IN": {
        "project_label": "Total fertility rate",
        "theme_group": "fertility",
        "narrative_role": "core_outcome"
    },
    "SP.URB.TOTL.IN.ZS": {
        "project_label": "Urban population (%)",
        "theme_group": "urbanization",
        "narrative_role": "context"
    },
}

dim_indicator_enriched = dim_indicator_clean.copy()
dim_indicator_enriched["project_label"] = dim_indicator_enriched["code"].map(
    lambda x: indicator_enrichment.get(x, {}).get("project_label")
)
dim_indicator_enriched["theme_group"] = dim_indicator_enriched["code"].map(
    lambda x: indicator_enrichment.get(x, {}).get("theme_group")
)
dim_indicator_enriched["narrative_role"] = dim_indicator_enriched["code"].map(
    lambda x: indicator_enrichment.get(x, {}).get("narrative_role")
)

dim_indicator_enriched

,source,code,name,project_label,theme_group,narrative_role
0,worldbank,SP.DYN.TFRT.IN,"Fertility rate, total (births per woman)",Total fertility rate,fertility,core_outcome
1,worldbank,SP.ADO.TFRT,"Adolescent fertility rate (births per 1,000 wo...",Adolescent fertility,fertility,core_outcome
2,worldbank,SH.STA.MMRT.NE,"Maternal mortality ratio (per 100,000 live bir...",Maternal mortality,health,health_outcome
3,worldbank,SL.TLF.CACT.FE.ZS,"Labor force participation rate, female (%)",Female labor force participation,labor_gender,gender_context
4,worldbank,NY.GDP.PCAP.CD,GDP per capita (current US$),GDP per capita,economy,context
5,worldbank,SP.URB.TOTL.IN.ZS,Urban population (% of total),Urban population (%),urbanization,context
6,worldbank,SE.SEC.ENRR.FE,"School enrollment, secondary, female (%)",Female secondary enrollment,education,gender_context
7,worldbank,SH.XPD.CHEX.GD.ZS,Current health expenditure (% of GDP),Health expenditure (% GDP),health_system,context
8,un,UN_18,Mean age of childbearing (5-year),Mean age of childbearing,fertility_timing,context
9,un,UN_67,Median age of population,Median age,population_structure,context


## 12. Визуальный обзор тематических групп

In [16]:
theme_counts = (
    dim_indicator_enriched["theme_group"]
    .value_counts(dropna=False)
    .rename_axis("theme_group")
    .reset_index(name="n_indicators")
)

theme_counts

,theme_group,n_indicators
0,population_structure,4
1,fertility,2
2,family_planning,2
3,health,1
4,labor_gender,1
5,economy,1
6,urbanization,1
7,education,1
8,health_system,1
9,fertility_timing,1


In [17]:
fig = px.bar(
    theme_counts,
    x="theme_group",
    y="n_indicators",
    title="Распределение индикаторов по тематическим группам",
    text="n_indicators"
)
fig.update_layout(xaxis_title="Тематическая группа", yaxis_title="Количество индикаторов")
fig.show()

## 13. Сохранение результатов

In [18]:
dim_indicator_clean.to_csv(OUTPUT_DIR / "dim_indicator_clean.csv", index=False)
dim_indicator_enriched.to_csv(OUTPUT_DIR / "dim_indicator_enriched.csv", index=False)

print("Файлы сохранены в:", OUTPUT_DIR)

Файлы сохранены в: notebooks\eda_outputs


## 14. Итоговые выводы

Таблица отражает основные аспекты которые влияют на фертильность и может использоваться для последующего анализа